# **`RAINFALL TIME SERIES FORECASTING`**

## **Business Problem Understanding**

**Goals** : 
- Predicting the Month-wise Rain Fall for next one year
- Predicting the Maximum amount of Rain fall in a single Day in a Month.

**Target Variables** : 
- Maximum Rainfall in a day
- Total Rainfall in a Month

**ML Problem Type** : Time Series Forecasting



### Import Necessary Libraries

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
mean_absolute_error,
mean_squared_error,
r2_score
)

# Time Series Model
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf
from statsmodels.tsa.arima.model import ARIMA

# Statistics
from scipy.stats import skew,kurtosis

import warnings
warnings.filterwarnings('ignore')

print(f'✅ Importing all Necessary Libraries.')

✅ Importing all Necessary Libraries.


### Load the Datasets

- load all the Datasets and combined all the data into one DataFrame

In [2]:
rf_monthly_highest_daily_df = pd.read_csv("rainfall-monthly-highest-daily-total.csv")

rf_monthly_no_of_rain_days_df = pd.read_csv("rainfall-monthly-number-of-rain-days.csv")

rf_monthly_total_df = pd.read_csv("rainfall-monthly-total.csv")


**Rainfall Monthly Highest Daily Record Display**

In [3]:
rf_monthly_highest_daily_df.head()

,month,maximum_rainfall_in_a_day
0,1982-01,36.5
1,1982-02,9.4
2,1982-03,61.7
3,1982-04,45.1
4,1982-05,33.0


**Rainfall Monthly Number of Rain Fall Days Display**

In [4]:
rf_monthly_no_of_rain_days_df.head()

,month,no_of_rainy_days
0,1982-01,10
1,1982-02,5
2,1982-03,11
3,1982-04,14
4,1982-05,10


**Rainfall Monthly Total Display**

In [5]:
rf_monthly_total_df.head()

,month,total_rainfall
0,1982-01,107.1
1,1982-02,27.8
2,1982-03,160.8
3,1982-04,157.0
4,1982-05,102.2


**Merge all the Dataframes into a One Final DataFrame**

In [6]:
df = pd.merge(rf_monthly_highest_daily_df,rf_monthly_no_of_rain_days_df,on="month")


df = pd.merge(df,rf_monthly_total_df,on="month")


print(f'✅ Final Merged DataFrame : \n')

df.head()

✅ Final Merged DataFrame : 



,month,maximum_rainfall_in_a_day,no_of_rainy_days,total_rainfall
0,1982-01,36.5,10,107.1
1,1982-02,9.4,5,27.8
2,1982-03,61.7,11,160.8
3,1982-04,45.1,14,157.0
4,1982-05,33.0,10,102.2


In [7]:
rows,cols = df.shape

print(f'Dataset Contains {rows} Rows * {cols} Columns.')

Dataset Contains 462 Rows * 4 Columns.


In [8]:
# General Information about the Dataset

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 462 entries, 0 to 461
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   month                      462 non-null    str    
 1   maximum_rainfall_in_a_day  462 non-null    float64
 2   no_of_rainy_days           462 non-null    int64  
 3   total_rainfall             462 non-null    float64
dtypes: float64(2), int64(1), str(1)
memory usage: 17.7 KB


In [9]:
# Statistical summary about the Dataset

df.describe().T

,count,mean,std,min,25%,50%,75%,max
maximum_rainfall_in_a_day,462.0,52.358225,35.685122,0.2,30.625,43.45,63.15,216.2
no_of_rainy_days,462.0,13.958874,4.947494,1.0,11.000,14.00,18.00,27.0
total_rainfall,462.0,176.236580,113.296793,0.2,89.025,158.45,237.95,765.9


**Missing Values**

In [10]:
df.isnull().sum()

month                        0
maximum_rainfall_in_a_day    0
no_of_rainy_days             0
total_rainfall               0
dtype: int64

**Duplicate Values Finding**

In [11]:
df.duplicated().sum()

np.int64(0)

**Numerical Columns Statistical Distributions**

In [12]:
num_columns = df.select_dtypes(include=np.number).columns.tolist()

print(f'Numerical Columns : \n{num_columns}\n')

for col in num_columns:
    skewness = skew(df[col].dropna())
    kurt = kurtosis(df[col].dropna())
    print(f'{col} → Skewness : {skewness} | Kurtosis : {kurt}\n')

Numerical Columns : 
['maximum_rainfall_in_a_day', 'no_of_rainy_days', 'total_rainfall']

maximum_rainfall_in_a_day → Skewness : 1.856151089297329 | Kurtosis : 4.290570168232787

no_of_rainy_days → Skewness : -0.1513903339642466 | Kurtosis : -0.4793803551044822

total_rainfall → Skewness : 1.2332292877991786 | Kurtosis : 2.314329780137431



In [13]:
# Converting Data type from String to Data and Time

df['month'] = pd.to_datetime(df['month'])

# Sort the Date
df = df.sort_values('month',ascending=True)

# Extracting Year and Month
df['year'] = df['month'].dt.year
df['Month'] = df['month'].dt.month
df['day'] = df['month'].dt.day


In [14]:
df.head()

,month,maximum_rainfall_in_a_day,no_of_rainy_days,total_rainfall,year,Month,day
0,1982-01-01,36.5,10,107.1,1982,1,1
1,1982-02-01,9.4,5,27.8,1982,2,1
2,1982-03-01,61.7,11,160.8,1982,3,1
3,1982-04-01,45.1,14,157.0,1982,4,1
4,1982-05-01,33.0,10,102.2,1982,5,1


### `🗒️Complete Data Analysis Report`

- Dataset Contains 462 Rows * 4 Columns.
- Dataset has **3** Numerical Columns and **1** Datatime Column Datatypes
- It has no Missing values
- It has NO Duplicate Values 